<a href="https://colab.research.google.com/github/Abandonalo/SpatialGenUnity/blob/main/notebooks/Colab_ComfyUI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# sanity check
import os
os.getcwd()

Mounted at /content/drive


'/content'

In [2]:
%cd /content/drive/MyDrive

import os
import subprocess
import sys

# Clone ComfyUI
if not os.path.exists("ComfyUI"):
    !git clone https://github.com/comfyanonymous/ComfyUI.git

%cd /content/drive/MyDrive/ComfyUI


!rm -f .git/index.lock

UPDATE_COMFY = False  # Set True only when you intentionally want to update ComfyUI.
if UPDATE_COMFY:
    subprocess.run(
        ["git", "stash", "push", "--include-untracked", "-m", "spatialgen-colab-autostash"],
        check=False,
    )
    subprocess.run(["git", "pull", "--ff-only"], check=True)
else:
    print("ℹ️ Skipping ComfyUI git pull for reproducible Colab startup")

# ----------------------------
# CUDA 12.6 check
# ----------------------------
def is_torch_cu126():
    try:
        import torch

        print("Found torch:", torch.__version__)
        print("CUDA build:", torch.version.cuda)

        return (
            torch.version.cuda == "12.6"
            and "+cu126" in torch.__version__
        )

    except Exception as e:
        print("Torch check failed:", e)
        return False


if is_torch_cu126():
    print("✅ CUDA 12.6 already installed — skipping")
else:
    print("⚠️ Installing CUDA 12.6 torch")

    !pip uninstall -y torch torchvision torchaudio

    !pip install \
        torch torchvision torchaudio \
        --index-url https://download.pytorch.org/whl/cu126

# Dependencies
!pip install -q -r requirements.txt
!pip install -q rembg onnxruntime

# Verify
import torch

print("✅ CUDA available:", torch.cuda.is_available())
print("💡 GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")
print("🔥 Torch CUDA:", torch.version.cuda)
print("📦 Torch:", torch.__version__)

/content/drive/MyDrive
/content/drive/MyDrive/ComfyUI
Saved working directory and index state WIP on master: 589228e6 Add slice_cond and per-model context window cond resizing (#12645)
Updating 589228e6..ec1896ac
error: Your local changes to the following files would be overwritten by merge:
	.ci/windows_amd_base_files/run_amd_gpu_disable_smart_memory.bat
	comfy/samplers.py
Please commit your changes or stash them before you merge.
Aborting
Found torch: 2.12.0+cu126
CUDA build: 12.6
✅ CUDA 12.6 already installed — skipping
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.7/21.7 MB 62.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 76.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.8/80.8 MB 26.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.6/42.6 MB 40.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.3/90.3 MB 24.7 MB/s eta 0:00:0

In [12]:
import os
import re
import shutil
import subprocess
import tarfile
import urllib.request

# ============================================================
# Config
# ============================================================

ZROK_VERSION = "1.1.11"
ZROK_ENABLE_TOKEN = os.environ.get("ZROK_ENABLE_TOKEN", "").strip()
if not ZROK_ENABLE_TOKEN:
    try:
        from google.colab import userdata
        ZROK_ENABLE_TOKEN = (userdata.get("ZROK_ENABLE_TOKEN") or "").strip()
    except Exception:
        ZROK_ENABLE_TOKEN = ""
if not ZROK_ENABLE_TOKEN:
    import getpass
    ZROK_ENABLE_TOKEN = getpass.getpass("Enter zrok enable token: ").strip()
if not ZROK_ENABLE_TOKEN:
    raise RuntimeError("ZROK_ENABLE_TOKEN is required to create the Colab tunnel.")

archive_name = f"zrok_{ZROK_VERSION}_linux_amd64.tar.gz"
archive_url = (
    f"https://github.com/openziti/zrok/releases/download/"
    f"v{ZROK_VERSION}/{archive_name}"
)

extract_dir = "zrok_extract"

cache_dir = "/content/drive/MyDrive/.cache/zrok"
cache_binary = os.path.join(
    cache_dir,
    f"zrok_{ZROK_VERSION}"
)

local_binary = "./zrok"

os.makedirs(cache_dir, exist_ok=True)

# ============================================================
# Install / restore zrok binary
# ============================================================

if os.path.exists(cache_binary):
    print("✅ Reusing cached zrok binary from Drive")
    shutil.copy2(cache_binary, local_binary)

else:
    print("⬇️ Downloading zrok...")

    # Cleanup old artifacts
    for path in [archive_name, extract_dir]:
        if os.path.isdir(path):
            shutil.rmtree(path)
        elif os.path.exists(path):
            os.remove(path)

    urllib.request.urlretrieve(
        archive_url,
        archive_name
    )

    print("📦 Extracting zrok...")

    os.makedirs(extract_dir, exist_ok=True)

    with tarfile.open(
        archive_name,
        "r:gz"
    ) as tar:
        tar.extractall(extract_dir)

    zrok_binary = None

    for root, _, files in os.walk(extract_dir):
        if "zrok" in files:
            zrok_binary = os.path.join(root, "zrok")
            break

    if zrok_binary is None:
        raise FileNotFoundError(
            "Could not locate zrok binary after extraction."
        )

    shutil.copy2(zrok_binary, local_binary)
    shutil.copy2(zrok_binary, cache_binary)

# Make executable
os.chmod(local_binary, 0o755)

if os.path.exists(cache_binary):
    os.chmod(cache_binary, 0o755)

# Verify zrok exists
subprocess.run([local_binary, "version"])

# ============================================================
# Enable zrok
# ============================================================

def enable_zrok(force=False):
    zrok_home = os.path.expanduser("~/.zrok")

    def reset_env():
        if os.path.exists(zrok_home):
            shutil.rmtree(zrok_home)

    # Optional clean reset
    if force:
        print("🧹 Resetting zrok environment...")
        reset_env()

    print("🔐 Validating zrok auth...")

    # --------------------------------------------------------
    # Try enable
    # --------------------------------------------------------

    result = subprocess.run(
        [local_binary, "enable", ZROK_ENABLE_TOKEN],
        capture_output=True,
        text=True,
    )

    combined = (
        (result.stdout or "")
        + "\n"
        + (result.stderr or "")
    ).strip()

    lowered = combined.lower()

    enable_ok = (
        result.returncode == 0
        or "already enabled" in lowered
        or "already have an enabled environment" in lowered
    )

    # --------------------------------------------------------
    # Broken Colab auth (/dev/tty) or stale state
    # --------------------------------------------------------

    if (
        "/dev/tty" in lowered
        or "no such device or address" in lowered
        or not enable_ok
    ):
        print(
            "⚠️ Broken or stale zrok auth detected — resetting"
        )

        reset_env()

        retry = subprocess.run(
            [local_binary, "enable", ZROK_ENABLE_TOKEN],
            capture_output=True,
            text=True,
        )

        retry_output = (
            (retry.stdout or "")
            + "\n"
            + (retry.stderr or "")
        ).strip()

        retry_lowered = retry_output.lower()

        enable_ok = (
            retry.returncode == 0
            or "already enabled" in retry_lowered
            or "already have an enabled environment"
            in retry_lowered
        )

        if not enable_ok:
            print("❌ zrok enable failed:")
            print(retry_output)
            return False

    # --------------------------------------------------------
    # Validate public share permission
    # --------------------------------------------------------

    print("🧪 Testing zrok share permission...")

    try:
        test_share = subprocess.run(
            [
                local_binary,
                "share",
                "public",
                "127.0.0.1:1"
            ],
            capture_output=True,
            text=True,
            timeout=20,
        )

        share_output = (
            (test_share.stdout or "")
            + "\n"
            + (test_share.stderr or "")
        ).strip()

        share_lower = share_output.lower()

        # Unauthorized token/environment
        if (
            "shareunauthorized" in share_lower
            or "401" in share_lower
        ):
            print(
                "⚠️ zrok unauthorized — "
                "resetting auth"
            )

            reset_env()

            retry = subprocess.run(
                [
                    local_binary,
                    "enable",
                    ZROK_ENABLE_TOKEN
                ],
                capture_output=True,
                text=True,
            )

            retry_output = (
                (retry.stdout or "")
                + "\n"
                + (retry.stderr or "")
            ).strip()

            retry_lower = retry_output.lower()

            retry_ok = (
                retry.returncode == 0
                or "already enabled" in retry_lower
                or "already have an enabled environment"
                in retry_lower
            )

            if not retry_ok:
                print("❌ zrok re-enable failed:")
                print(retry_output)
                return False

            print("✅ zrok re-enabled")

        else:
            print("✅ zrok ready")

    except subprocess.TimeoutExpired:
        print("⚠️ Share test timed out")
        print("✅ Assuming zrok ready")

    return True



enable_zrok()

✅ Reusing cached zrok binary from Drive
🔐 Validating zrok auth...
🧪 Testing zrok share permission...
⚠️ zrok unauthorized — resetting auth
✅ zrok re-enabled


True

In [9]:
# ==========================================
# Hunyuan3D 2.1 + Essentials Custom Nodes
# ==========================================

import os
import shutil
import sys
import subprocess

COMFY_PATH = "/content/drive/MyDrive/ComfyUI"
CUSTOM_NODES = f"{COMFY_PATH}/custom_nodes"
HY3D_NODE = f"{CUSTOM_NODES}/ComfyUI-Hunyuan3d-2-1"
ESSENTIALS_NODE = f"{CUSTOM_NODES}/ComfyUI-essentials"
CUSTOM_NODE_SETUP_STAMP = "/tmp/spatialgen_hunyuan_setup_v1"
FORCE_CUSTOM_NODE_SETUP = False

print("🚀 Setting up ComfyUI custom nodes...")


def run(cmd):
    return subprocess.run(cmd, check=False)


def install_requirements_if_present(path, label):
    req_path = os.path.join(path, "requirements.txt")
    if os.path.exists(req_path):
        print(f"📦 Installing {label} dependencies...")
        run([sys.executable, "-m", "pip", "install", "-r", req_path])
    else:
        print(f"ℹ️ {label} has no requirements.txt")


def install_wheel_or_source(package_dir, label):
    dist_dir = os.path.join(package_dir, "dist")
    if os.path.exists(dist_dir):
        for file in os.listdir(dist_dir):
            if "linux" in file and file.endswith(".whl"):
                wheel_path = os.path.join(dist_dir, file)
                print(f"⚡ Installing {label} wheel: {file}")
                result = run([sys.executable, "-m", "pip", "install", wheel_path])
                if result.returncode == 0:
                    return
    print(f"🔧 Falling back to source build ({label})...")
    run([sys.executable, "-m", "pip", "install", package_dir])


if os.path.exists(CUSTOM_NODE_SETUP_STAMP) and not FORCE_CUSTOM_NODE_SETUP:
    print("✅ Reusing custom-node setup from this runtime")
else:
    # Install system build tools only if they are missing.
    if shutil.which("g++") and shutil.which("make"):
        print("✅ Build tools already available")
    else:
        print("🔧 Installing system build dependencies...")
        run(["apt-get", "update"])
        run(["apt-get", "install", "-y", "build-essential"])

    if not os.path.exists(HY3D_NODE):
        print("📦 Cloning Hunyuan3D 2.1...")
        run(["git", "clone", "https://github.com/visualbruno/ComfyUI-Hunyuan3d-2-1", HY3D_NODE])
    else:
        print("✅ Hunyuan3D already exists")

    if not os.path.exists(ESSENTIALS_NODE):
        print("📦 Cloning ComfyUI Essentials...")
        run(["git", "clone", "https://github.com/cubiq/ComfyUI_essentials", ESSENTIALS_NODE])
    else:
        print("✅ Essentials already exists")

    install_requirements_if_present(HY3D_NODE, "Hunyuan")
    install_requirements_if_present(ESSENTIALS_NODE, "Essentials")

    print("🧠 Installing custom rasterizer...")
    install_wheel_or_source(f"{HY3D_NODE}/hy3dpaint/custom_rasterizer", "rasterizer")

    print("🎨 Installing differentiable renderer...")
    install_wheel_or_source(f"{HY3D_NODE}/hy3dpaint/DifferentiableRenderer", "renderer")

    with open(CUSTOM_NODE_SETUP_STAMP, "w", encoding="utf-8") as f:
        f.write("ok")

    print("✅ All custom nodes installed successfully!")

🚀 Setting up ComfyUI custom nodes...
✅ Build tools already available
✅ Hunyuan3D already exists
✅ Essentials already exists
📦 Installing Hunyuan dependencies...
📦 Installing Essentials dependencies...
🧠 Installing custom rasterizer...
⚡ Installing rasterizer wheel: custom_rasterizer-0.1-cp311-cp311-linux_x86_64.whl
🔧 Falling back to source build (rasterizer)...
🎨 Installing differentiable renderer...
⚡ Installing renderer wheel: mesh_inpaint_processor-0.0.0-cp311-cp311-linux_x86_64.whl
🔧 Falling back to source build (renderer)...
✅ All custom nodes installed successfully!


In [ ]:
!pkill -f main.py

In [13]:
import os
import subprocess
import time
import torch

os.environ["CUDA_VISIBLE_DEVICES"] = "0"

# Ensure PyTorch defaults to CUDA
if torch.cuda.is_available():
    torch.set_default_device("cuda")
    print("✅ Using GPU:", torch.cuda.get_device_name(0))
else:
    print("❌ CUDA not available — something is wrong")

os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["CUDA_LAUNCH_BLOCKING"] = "0"

DISABLE_SMART_MEMORY = False
FORCE_FP32 = False

launch_args = [
    "python",
    "main.py",
    "--listen", "0.0.0.0",
    "--port", "8188",
]
if DISABLE_SMART_MEMORY:
    launch_args.append("--disable-smart-memory")
if FORCE_FP32:
    launch_args.append("--force-fp32")

comfy_process = subprocess.Popen(
    launch_args,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1  # real-time logs
)

ready = False
for _ in range(120):
    line = comfy_process.stdout.readline()

    if line:
        print(line.strip())

        if "Starting server" in line or "To see the GUI go to" in line:
            ready = True
            break

    time.sleep(0.5)

if ready:
    print("✅ ComfyUI is running on port 8188")
else:
    print("❌ ComfyUI may not have started correctly")


✅ Using GPU: Tesla T4
Found comfy_kitchen backend eager: {'available': True, 'disabled': False, 'unavailable_reason': None, 'capabilities': ['apply_rope', 'apply_rope1', 'dequantize_mxfp8', 'dequantize_nvfp4', 'dequantize_per_tensor_fp8', 'gemv_awq_w4a16', 'quantize_mxfp8', 'quantize_nvfp4', 'quantize_per_tensor_fp8', 'quantize_svdquant_w4a4', 'scaled_mm_mxfp8', 'scaled_mm_nvfp4', 'scaled_mm_svdquant_w4a4', 'stochastic_rounding_fp8']}
Found comfy_kitchen backend triton: {'available': True, 'disabled': True, 'unavailable_reason': None, 'capabilities': ['apply_rope', 'apply_rope1', 'dequantize_nvfp4', 'dequantize_per_tensor_fp8', 'quantize_mxfp8', 'quantize_nvfp4', 'quantize_per_tensor_fp8']}
Found comfy_kitchen backend cuda: {'available': True, 'disabled': True, 'unavailable_reason': None, 'capabilities': ['apply_rope', 'apply_rope1', 'dequantize_nvfp4', 'dequantize_per_tensor_fp8', 'gemv_awq_w4a16', 'quantize_mxfp8', 'quantize_nvfp4', 'quantize_per_tensor_fp8', 'quantize_svdquant_w4a4'

In [ ]:
print(
    "Skipping direct ComfyUI zrok share on port 8188. "
    "Unity uses the FastAPI proxy on port 8000; run the proxy and reserved zrok cells below."
)

╭──────────────────────────────────╮╭──────────────────────────╮
│https://jj4ez65bowie.share.zrok.io││     [PUBLIC] [PROXY]     │
╰──────────────────────────────────╯╰──────────────────────────╯
╭╮                                                              
││                                                              
╰╯                                                              

In [ ]:
from fastapi import FastAPI, HTTPException, Query, Request, Response
import base64
import copy
import json
import os
import random
import re
import requests
import time
import uuid

app = FastAPI()

COMFYUI_URL = "http://127.0.0.1:8188"
WORKFLOW_DIR = os.path.join(
    COMFY_PATH,
    "user",
    "default",
    "workflows",
)
FULL_WORKFLOW_TEMPLATE_PATH = os.path.join(WORKFLOW_DIR, "Full_Workflow_API.json")
UPLOAD_IMAGE_WORKFLOW_TEMPLATE_PATH = os.path.join(WORKFLOW_DIR, "Upload_Image_API.json")
COMFY_INPUT_DIR = os.path.join(COMFY_PATH, "input")


def load_workflow_template(path):
    if not os.path.exists(path):
        return None

    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def is_ui_workflow(workflow):
    return isinstance(workflow, dict) and "nodes" in workflow and "links" in workflow


def has_uploaded_asset_image(body):
    asset_image = body.get("asset_image")
    return isinstance(asset_image, dict) and bool(asset_image.get("image_base64"))


def uses_image_workflow(body):
    return str(body.get("input_mode", "")).lower() == "image" or has_uploaded_asset_image(body)


def load_selected_workflow(body):
    workflow_path = UPLOAD_IMAGE_WORKFLOW_TEMPLATE_PATH if uses_image_workflow(body) else FULL_WORKFLOW_TEMPLATE_PATH
    workflow = load_workflow_template(workflow_path)
    if workflow is None:
        raise HTTPException(
            status_code=400,
            detail=(
                "Workflow not found on Google Drive. Save "
                f"{os.path.basename(workflow_path)} to {WORKFLOW_DIR} before using /generate."
            ),
        )

    if is_ui_workflow(workflow):
        raise HTTPException(
            status_code=400,
            detail=(
                f"{os.path.basename(workflow_path)} is in ComfyUI UI format, not API format. "
                "Export/save the workflow in API format before using /generate."
            ),
        )

    return copy.deepcopy(workflow)


POSITIVE_PROMPT_NODE_IDS = {"55"}
NEGATIVE_PROMPT_NODE_IDS = {"56"}


def apply_prompt_inputs(workflow, prompt, negative_prompt=""):
    for node_id, node in workflow.items():
        if not isinstance(node, dict) or node.get("class_type") != "CLIPTextEncode":
            continue

        meta = node.get("_meta") or {}
        title = str(meta.get("title", "")).lower()
        inputs = node.setdefault("inputs", {})

        if str(node_id) in NEGATIVE_PROMPT_NODE_IDS:
            inputs["text"] = negative_prompt
        elif str(node_id) in POSITIVE_PROMPT_NODE_IDS:
            inputs["text"] = prompt
        elif "negative" in title:
            inputs["text"] = negative_prompt
        else:
            inputs["text"] = prompt


def apply_generation_inputs(workflow, generation):
    if not generation:
        return

    seed = generation.get("seed")
    steps = generation.get("steps")
    cfg = generation.get("cfg")
    sampler = generation.get("sampler")
    width = generation.get("width")
    height = generation.get("height")

    # Unity uses -1 to mean "pick a random seed"; ComfyUI validators require >= 0.
    if seed is None or int(seed) < 0:
        seed = random.randint(0, 2**31 - 1)

    for node in workflow.values():
        if not isinstance(node, dict):
            continue

        class_type = node.get("class_type")
        inputs = node.setdefault("inputs", {})

        if class_type == "KSampler":
            inputs["seed"] = int(seed)
            if steps is not None:
                inputs["steps"] = steps
            if cfg is not None:
                inputs["cfg"] = cfg
            if sampler:
                inputs["sampler_name"] = sampler
        elif class_type == "EmptyLatentImage":
            if width is not None:
                inputs["width"] = width
            if height is not None:
                inputs["height"] = height


def sanitize_upload_filename(file_name, request_id):
    base_name = os.path.basename((file_name or "").strip())
    if not base_name:
        base_name = "proxy_image.png"

    stem, ext = os.path.splitext(base_name)
    stem = re.sub(r"[^A-Za-z0-9._-]+", "_", stem).strip("._-") or "proxy_image"
    ext = ext.lower() if ext else ".png"
    if ext not in {".png", ".jpg", ".jpeg", ".webp"}:
        ext = ".png"

    return f"{request_id}_{stem}{ext}"


def persist_uploaded_asset_image(asset_image, request_id):
    if not isinstance(asset_image, dict):
        return None

    image_base64 = asset_image.get("image_base64") or ""
    if not image_base64:
        raise HTTPException(status_code=400, detail="Image workflow selected, but asset_image.image_base64 is empty.")

    try:
        image_bytes = base64.b64decode(image_base64, validate=True)
    except Exception as exc:
        raise HTTPException(status_code=400, detail=f"Invalid asset_image.image_base64 payload: {exc}") from exc

    if not image_bytes:
        raise HTTPException(status_code=400, detail="Decoded asset image is empty.")

    os.makedirs(COMFY_INPUT_DIR, exist_ok=True)
    image_name = sanitize_upload_filename(asset_image.get("file_name"), request_id)
    image_path = os.path.join(COMFY_INPUT_DIR, image_name)
    with open(image_path, "wb") as f:
        f.write(image_bytes)
    return image_name


def apply_uploaded_image_inputs(workflow, image_name):
    if not image_name:
        return

    applied = False
    for node in workflow.values():
        if not isinstance(node, dict):
            continue

        class_type = node.get("class_type")
        if class_type not in {"Hy3D21LoadImageWithTransparency", "LoadImage"}:
            continue

        inputs = node.setdefault("inputs", {})
        inputs["image"] = image_name
        inputs["upload"] = "image"
        applied = True

    if not applied:
        raise HTTPException(
            status_code=400,
            detail="Upload_Image_API.json does not contain a supported image loader node.",
        )


def build_workflow(body):
    # In proxy mode the server owns workflow selection. Unity sends intent only.
    workflow = load_selected_workflow(body)
    apply_prompt_inputs(
        workflow,
        body.get("prompt", ""),
        body.get("negative_prompt", ""),
    )
    apply_generation_inputs(workflow, body.get("generation") or {})

    if uses_image_workflow(body):
        image_name = persist_uploaded_asset_image(body.get("asset_image"), body.get("request_id") or str(uuid.uuid4()))
        apply_uploaded_image_inputs(workflow, image_name)

    return workflow


def fetch_history(prompt_id):
    response = requests.get(f"{COMFYUI_URL}/history/{prompt_id}", timeout=30)
    response.raise_for_status()
    return response.json()


def fetch_queue():
    response = requests.get(f"{COMFYUI_URL}/queue", timeout=30)
    response.raise_for_status()
    return response.json()


def get_history_entry(history_payload, prompt_id):
    if isinstance(history_payload, dict):
        if prompt_id in history_payload:
            return history_payload[prompt_id]
        if "outputs" in history_payload or "status" in history_payload:
            return history_payload
    return None


def get_queue_state(queue_payload, prompt_id):
    running = queue_payload.get("queue_running") if isinstance(queue_payload, dict) else None
    pending = queue_payload.get("queue_pending") if isinstance(queue_payload, dict) else None

    def contains_prompt(entries):
        if not isinstance(entries, list):
            return False
        needle = str(prompt_id)
        for entry in entries:
            if needle in json.dumps(entry, default=str):
                return True
        return False

    if contains_prompt(running):
        return "running"
    if contains_prompt(pending):
        return "pending"
    return "missing"


def extract_execution_error(history_entry):
    if not isinstance(history_entry, dict):
        return ""

    status = history_entry.get("status") or {}
    messages = status.get("messages") or []
    for message in messages:
        if not isinstance(message, list) or len(message) < 2:
            continue
        event_type, payload = message[0], message[1]
        if event_type != "execution_error" or not isinstance(payload, dict):
            continue
        return payload.get("exception_message") or payload.get("exception_type") or ""

    return ""


def extract_output_files(outputs):
    files = []
    seen = set()

    for node_id, node_output in outputs.items():
        if not isinstance(node_output, dict):
            continue

        for output_key, output_value in node_output.items():
            candidates = output_value if isinstance(output_value, list) else [output_value]
            for item in candidates:
                if not isinstance(item, dict) or "filename" not in item:
                    continue

                ref = {
                    "filename": item["filename"],
                    "subfolder": item.get("subfolder", ""),
                    "type": item.get("type", "output"),
                    "node_id": str(node_id),
                    "output_key": str(output_key),
                }
                dedupe_key = (ref["filename"], ref["subfolder"], ref["type"])
                if dedupe_key in seen:
                    continue
                seen.add(dedupe_key)
                files.append(ref)

    def sort_key(ref):
        ext = os.path.splitext(ref["filename"])[1].lower()
        if ext in {".glb", ".gltf", ".obj", ".fbx"}:
            return (0, ref["node_id"], ref["output_key"], ref["filename"])
        if ext in {".png", ".jpg", ".jpeg", ".webp"}:
            return (1, ref["node_id"], ref["output_key"], ref["filename"])
        return (2, ref["node_id"], ref["output_key"], ref["filename"])

    return sorted(files, key=sort_key)


def extract_images(files):
    return [
        ref for ref in files
        if os.path.splitext(ref["filename"])[1].lower() in {".png", ".jpg", ".jpeg", ".webp"}
    ]


def extract_meshes(files):
    return [
        ref for ref in files
        if os.path.splitext(ref["filename"])[1].lower() in {".glb", ".gltf", ".obj", ".fbx"}
    ]


@app.get("/health")
def health():
    try:
        response = requests.get(f"{COMFYUI_URL}/system_stats", timeout=10)
        response.raise_for_status()
        return {"status": "ok", "comfyui": "reachable"}
    except requests.RequestException as exc:
        raise HTTPException(status_code=503, detail=f"ComfyUI unreachable: {exc}") from exc


@app.post("/generate")
async def generate(req: Request):
    body = await req.json()

    request_id = body.get("request_id") or str(uuid.uuid4())
    body["request_id"] = request_id
    start_time = time.time()

    try:
        workflow = build_workflow(body)
        response = requests.post(
            f"{COMFYUI_URL}/prompt",
            json={"prompt": workflow},
            timeout=60,
        )

        if not response.ok:
            detail = response.text
            print(f"[{request_id}] ComfyUI /prompt validation failed: {detail}")
            raise HTTPException(
                status_code=502,
                detail=f"ComfyUI /prompt rejected workflow: {detail}",
            )

        result = response.json()
        prompt_id = result.get("prompt_id")
        duration = time.time() - start_time

        if not prompt_id:
            raise HTTPException(status_code=502, detail=f"ComfyUI /prompt missing prompt_id: {result}")

        print(f"[{request_id}] queued prompt_id={prompt_id} ({duration:.2f}s)")

        return {
            "request_id": request_id,
            "status": "queued",
            "duration": duration,
            "prompt_id": prompt_id,
            "comfyui_response": result,
        }

    except HTTPException:
        raise
    except requests.RequestException as exc:
        duration = time.time() - start_time
        response_text = exc.response.text if exc.response is not None else str(exc)
        print(f"[{request_id}] request error: {response_text}")
        raise HTTPException(status_code=502, detail=f"Failed to submit prompt to ComfyUI: {response_text}") from exc
    except Exception as exc:
        duration = time.time() - start_time
        print(f"[{request_id}] unexpected error after {duration:.2f}s: {exc}")
        raise HTTPException(status_code=500, detail=str(exc)) from exc


@app.post("/refine")
async def refine(req: Request):
    body = await req.json()
    return {
        "requestId": body.get("requestId", ""),
        "refinedImageBase64": "",
        "meshBase64": "",
        "success": False,
        "errorMessage": (
            "Colab_ComfyUI.ipynb currently supports Unity /generate only. "
            "Use the local FastAPI backend for refinement, or port the refinement "
            "workflow templates into this notebook before using /refine."
        ),
    }


@app.post("/refine_multi_view")
async def refine_multi_view(req: Request):
    body = await req.json()
    return {
        "requestId": body.get("requestId", ""),
        "refinedViews": [],
        "meshBase64": "",
        "success": False,
        "errorMessage": (
            "Colab_ComfyUI.ipynb currently supports Unity /generate only. "
            "Multi-view refinement requires the local router's refinement graphs "
            "and has not been ported to this Colab notebook yet."
        ),
    }


@app.get("/result/{prompt_id}")
def get_result(prompt_id: str):
    try:
        history_payload = fetch_history(prompt_id)
        history_entry = get_history_entry(history_payload, prompt_id)
        queue_payload = fetch_queue()
        queue_state = get_queue_state(queue_payload, prompt_id)
        queue_running_count = len(queue_payload.get("queue_running") or []) if isinstance(queue_payload, dict) else 0
        queue_pending_count = len(queue_payload.get("queue_pending") or []) if isinstance(queue_payload, dict) else 0

        if history_entry is None:
            status = "running" if queue_state in {"running", "pending"} else "error"
            error_message = "Prompt is missing from both ComfyUI history and queue." if queue_state == "missing" else ""
            return {
                "status": status,
                "prompt_id": prompt_id,
                "completed": False,
                "exception_message": error_message,
                "queue_state": queue_state,
                "queue_running_count": queue_running_count,
                "queue_pending_count": queue_pending_count,
                "files": [],
                "meshes": [],
                "images": [],
                "history": {},
            }

        outputs = history_entry.get("outputs") or {}
        files = extract_output_files(outputs)
        meshes = extract_meshes(files)
        images = extract_images(files)
        completed = bool((history_entry.get("status") or {}).get("completed")) or bool(files)
        error_message = extract_execution_error(history_entry)
        status = "error" if error_message else ("success" if completed else "running")

        return {
            "status": status,
            "prompt_id": prompt_id,
            "completed": completed,
            "exception_message": error_message,
            "queue_state": queue_state,
            "queue_running_count": queue_running_count,
            "queue_pending_count": queue_pending_count,
            "files": files,
            "meshes": meshes,
            "images": images,
            "history": {prompt_id: history_entry},
        }
    except requests.RequestException as exc:
        raise HTTPException(status_code=502, detail=f"Failed to query ComfyUI history: {exc}") from exc


@app.get("/history/{prompt_id}")
def proxy_history(prompt_id: str):
    try:
        return fetch_history(prompt_id)
    except requests.RequestException as exc:
        raise HTTPException(status_code=502, detail=f"Failed to query ComfyUI history: {exc}") from exc


@app.get("/view")
def proxy_view(
    filename: str = Query(...),
    subfolder: str = Query(""),
    type: str = Query("output"),
):
    try:
        response = requests.get(
            f"{COMFYUI_URL}/view",
            params={
                "filename": filename,
                "subfolder": subfolder,
                "type": type,
            },
            timeout=60,
        )
        response.raise_for_status()
        return Response(
            content=response.content,
            media_type=response.headers.get("content-type", "application/octet-stream"),
        )
    except requests.RequestException as exc:
        raise HTTPException(status_code=502, detail=f"Failed to fetch ComfyUI output file: {exc}") from exc


In [ ]:
import uvicorn
from threading import Thread

def run():
    uvicorn.run(app, host="0.0.0.0", port=8000)

thread = Thread(target=run)
thread.start()

print("✅ FastAPI proxy running on port 8000")


✅ FastAPI proxy running on port 8000


INFO:     Started server process [3890]
INFO:     Waiting for application startup.
INFO:     Application startup complete.


In [ ]:
import json
import re
import subprocess
import time

TARGET = "http://127.0.0.1:8000"
RESERVED_NAME = "comfyuitunnel"


def extract_url(text):
    match = re.search(r'https://[^\s]+', text or "")
    return match.group(0) if match else None


def run_zrok_command(args):
    result = subprocess.run(
        args,
        capture_output=True,
        text=True,
        check=False,
    )
    combined = ((result.stdout or "") + "\n" + (result.stderr or "")).strip()
    if combined:
        print(combined)
    return result, combined


def is_unauthorized(text):
    lowered = (text or "").lower()
    return "unauthorized" in lowered or "401" in lowered


def ensure_zrok_auth(force=False):
    print("🔑 Ensuring zrok auth is valid...")
    if enable_zrok(force=force):
        return True
    print("❌ zrok enable failed")
    return False


def reserve_zrok_share():
    print(f"🔒 Ensuring reserved zrok share '{RESERVED_NAME}' points to {TARGET}...")
    result, combined = run_zrok_command(['./zrok', 'reserve', 'public', TARGET, '-n', RESERVED_NAME])

    if result.returncode == 0:
        return True

    lowered = combined.lower()
    if "reserved frontend endpoint" in lowered or "already" in lowered:
        return True

    if is_unauthorized(combined):
        print("⚠️ zrok auth appears stale. Re-enabling and retrying reservation...")
        if ensure_zrok_auth(force=True):
            retry_result, retry_combined = run_zrok_command(['./zrok', 'reserve', 'public', TARGET, '-n', RESERVED_NAME])
            retry_lowered = retry_combined.lower()
            if retry_result.returncode == 0 or "reserved frontend endpoint" in retry_lowered or "already" in retry_lowered:
                return True
            combined = retry_combined

    print("⚠️ Could not create or confirm reserved share. Will try a non-reserved public share instead.")
    return False


def start_reserved_zrok():
    return subprocess.Popen(
        [
            './zrok',
            'share',
            'reserved',
            RESERVED_NAME,
            '--headless'
        ],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True
    )


def start_public_zrok():
    return subprocess.Popen(
        [
            './zrok',
            'share',
            'public',
            TARGET,
            '--headless'
        ],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True
    )


def start_and_monitor(process_factory, allow_auth_retry):
    process = process_factory()
    public_url = None
    saw_unauthorized = False

    for _ in range(30):
        line = process.stdout.readline()
        if line:
            line = line.strip()
            print(line)

            if is_unauthorized(line):
                saw_unauthorized = True
                break

            try:
                payload = json.loads(line)
                msg = payload.get("msg", "")
                public_url = extract_url(msg) or extract_url(line)
            except json.JSONDecodeError:
                public_url = extract_url(line)

            if public_url:
                return public_url, False

        time.sleep(1)

    if saw_unauthorized and allow_auth_retry:
        print("⚠️ zrok share reported unauthorized. Re-enabling and retrying once...")
        if ensure_zrok_auth(force=True):
            return start_and_monitor(process_factory, allow_auth_retry=False)

    return None, saw_unauthorized


use_reserved = reserve_zrok_share()
public_url, saw_unauthorized = start_and_monitor(
    start_reserved_zrok if use_reserved else start_public_zrok,
    allow_auth_retry=True,
)

if not public_url and use_reserved and not saw_unauthorized:
    print("⚠️ Reserved share did not start cleanly. Falling back to non-reserved public share...")
    public_url, _ = start_and_monitor(start_public_zrok, allow_auth_retry=True)
    use_reserved = False

if not public_url:
    print("❌ Failed to start zrok")
else:
    print(f"\n🚀 PUBLIC ENDPOINT: {public_url}\n")
    if not use_reserved:
        print("ℹ️ Update Unity backend URL if you use this non-reserved endpoint.")


🔒 Ensuring reserved zrok share 'comfyuitunnel' points to http://127.0.0.1:8000...
{"file":"/__w/zrok/zrok/cmd/zrok/reserve.go:167","func":"main.(*reserveCommand).run","level":"info","msg":"your reserved share token is 'comfyuitunnel'","time":"2026-03-24T14:42:55.077Z"}
{"file":"/__w/zrok/zrok/cmd/zrok/reserve.go:169","func":"main.(*reserveCommand).run","level":"info","msg":"reserved frontend endpoint: https://comfyuitunnel.share.zrok.io","time":"2026-03-24T14:42:55.077Z"}
{"file":"/__w/zrok/zrok/cmd/zrok/shareReserved.go:132","func":"main.(*shareReservedCommand).shareLocal","level":"info","msg":"sharing target: 'http://127.0.0.1:8000'","time":"2026-03-24T14:42:56.411Z"}
{"file":"/__w/zrok/zrok/cmd/zrok/shareReserved.go:147","func":"main.(*shareReservedCommand).shareLocal","level":"info","msg":"using existing backend target: http://127.0.0.1:8000","time":"2026-03-24T14:42:56.412Z"}
{"file":"/__w/zrok/zrok/cmd/zrok/shareReserved.go:330","func":"main.(*shareReservedCommand).shareLocal","l

In [ ]:
import requests
print(requests.get("http://127.0.0.1:8188/system_stats", timeout=10).status_code)
print(requests.get("http://127.0.0.1:8000/health", timeout=10).text)

200
INFO:     127.0.0.1:34680 - "GET /health HTTP/1.1" 200 OK
{"status":"ok","comfyui":"reachable"}
